In [18]:
import os, gc, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16

In [19]:
def load_any_model(model_id: str, device: str = DEVICE, dtype=DTYPE):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map={"":0},
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    return model, tokenizer

#"llama31_8b": "meta-llama/Llama-3.1-8B",
#"llama32_3b": "meta-llama/Llama-3.2-3B",
#"mistral7b": "mistralai/Mistral-7B-v0.1",
#"qwen3_8b": "Qwen/Qwen3-8B",
#"olmo7b": "allenai/OLMo-7B"
MODEL_ID = "meta-llama/Llama-3.2-3B"
model, tokenizer = load_any_model(MODEL_ID)
print("✅ Loaded", MODEL_ID, "on", DEVICE)


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.51it/s]

✅ Loaded meta-llama/Llama-3.2-3B on cuda:0


In [ ]:
SUBJECT = "abstract_algebra"   # pick a subject for testing, default "all"
mmlu = load_dataset("cais/mmlu", SUBJECT)

print(mmlu)
print("Sample example:\n", mmlu["validation"][0])

DatasetDict({
    test: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 11
    })
    dev: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 5
    })
})
Sample example:
 {'question': 'The cyclic subgroup of Z_24 generated by 18 has order', 'subject': 'abstract_algebra', 'choices': ['4', '8', '12', '6'], 'answer': 0}


In [27]:
def build_prompt(question, choices):
    prompt = "Answer the following multiple-choice question with a single capital letter (A, B, C, or D).\n"
    prompt += "Format:\nAnswer: X\n\n"
    prompt += f"Question: {question}\n"
    for i, c in enumerate(choices):
        prompt += f"{chr(65+i)}. {c}\n"
    prompt += "Answer:"
    return prompt

In [28]:
@torch.inference_mode()
def generate_mcq_answer(model, tokenizer, question, choices, max_new_tokens=8):
    prompt = build_prompt(question, choices)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        use_cache=False
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # extract the answer letter if present
    tail = text.split("Answer:")[-1]
    for ch in tail:
        if ch in "ABCD":
            return ch, text
    return "", text

In [29]:
for i in range(10):
    ex = mmlu["validation"][i]
    pred, raw = generate_mcq_answer(model, tokenizer, ex["question"], ex["choices"])
    gold = chr(65 + ex["answer"])

    print("="*70)
    print("Q:", ex["question"])
    for j, c in enumerate(ex["choices"]):
        print(f"{chr(65+j)}. {c}")
    print("GT:", gold, "|", ex["choices"][ex["answer"]])
    print("Model raw output:\n", raw)
    print("Predicted letter:", pred)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

Q: The cyclic subgroup of Z_24 generated by 18 has order
A. 4
B. 8
C. 12
D. 6
GT: A | 4
Model raw output:
 Answer the following multiple-choice question with a single capital letter (A, B, C, or D).
Format:
Answer: X

Question: The cyclic subgroup of Z_24 generated by 18 has order
A. 4
B. 8
C. 12
D. 6
Answer: C
Predicted letter: C
Q: Find the order of the factor group Z_6/<3>.
A. 2
B. 3
C. 6
D. 12
GT: B | 3
Model raw output:
 Answer the following multiple-choice question with a single capital letter (A, B, C, or D).
Format:
Answer: X

Question: Find the order of the factor group Z_6/<3>.
A. 2
B. 3
C. 6
D. 12
Answer: B
Predicted letter: B
Q: Statement 1 | A permutation that is a product of m even permutations and n odd permutations is an even permutation if and only if n is even. Statement 2 | Every group is isomorphic to a group of permutations.
A. True, True
B. False, False
C. True, False
D. False, True
GT: A | True, True
Model raw output:
 Answer the following multiple-choice questio

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: Statement 1 | If a and b are elements of finite order in an Abelian group, then |ab| is the lcm (|a|,|b|). Statement 2 | If g is a group element and g^n = e, then |g| = n.
A. True, True
B. False, False
C. True, False
D. False, True
GT: B | False, False
Model raw output:
 Answer the following multiple-choice question with a single capital letter (A, B, C, or D).
Format:
Answer: X

Question: Statement 1 | If a and b are elements of finite order in an Abelian group, then |ab| is the lcm (|a|,|b|). Statement 2 | If g is a group element and g^n = e, then |g| = n.
A. True, True
B. False, False
C. True, False
D. False, True
Answer: A
Predicted letter: A
Q: Statement 1 | If f is a homomorphism from G to K and H is normal in G then f(H) is normal in K. Statement 2 | If f is a homomorphism from G to a group and H is finite subgroup of G, then |f(H)| divides |H|.
A. True, True
B. False, False
C. True, False
D. False, True
GT: D | False, True
Model raw output:
 Answer the following multiple-cho

In [30]:
def eval_acc_generate(model, tokenizer, ds, N=30):
    correct = 0
    for i in range(min(N, len(ds))):
        ex = ds[i]
        pred, raw = generate_mcq_answer(model, tokenizer, ex["question"], ex["choices"])
        gold = chr(65 + ex["answer"])
        correct += int(pred == gold)

        if (i+1) % 10 == 0:
            print(f"[{i+1}/{N}] running acc = {correct/(i+1):.3f}")
    total = min(N, len(ds))
    print(f"✅ Final acc: {correct}/{total} = {correct/total:.3f}")

# run on first 30 validation items
eval_acc_generate(model, tokenizer, mmlu["validation"], N=30)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more de

[10/30] running acc = 0.300
✅ Final acc: 3/11 = 0.273
